# Phase 2 — Feature Extraction
**Cross-Generator AI Image Detection**

This notebook:
1. Mounts Google Drive (images + output)
2. Clones the GitHub repository (source code)
3. Installs dependencies
4. Runs `extract_features.py` (the production runner)
5. Loads and verifies the saved outputs
6. Generates verification visualisations

> **All extraction logic lives in `src/extract_features.py`.**
> This notebook is for setup, execution, and post-run analysis only.


## Cell 1 — Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Verify the expected folder structure exists
import os
DRIVE_ROOT = "/content/drive/MyDrive/ML_Project"
assert os.path.isdir(DRIVE_ROOT), f"Drive root not found: {DRIVE_ROOT}"
assert os.path.isdir(f"{DRIVE_ROOT}/images"), "images/ folder not found on Drive"
assert os.path.isdir(f"{DRIVE_ROOT}/processed/splits"), "splits/ folder not found"
print("✅ Drive mounted and folder structure verified.")


## Cell 2 — Clone Repository & Install Dependencies


In [ ]:
# Replace with your actual GitHub repo URL
REPO_URL  = "https://github.com/<your-username>/cross-generator-ai-image-detection.git"
REPO_DIR  = "/content/repo"

!git clone {REPO_URL} {REPO_DIR}

# Install required packages
!pip install -q scikit-image PyWavelets tqdm joblib scipy

# Add src/ to Python path so imports work
import sys
sys.path.insert(0, f"{REPO_DIR}/src")

print("✅ Repository cloned and dependencies installed.")


## Cell 3 — Verify Imports


In [ ]:
# Quick import check before running the full pipeline
import numpy as np
import pandas as pd
import cv2
import pywt
import skimage
import scipy

from config   import IMAGE_SIZE, FEATURE_VERSION, GENERATORS, RANDOM_SEED
from utils    import prepare_image
from features import combine_features, FAMILY_ORDER
from dataset  import load_splits

print(f"✅ All imports successful.")
print(f"   IMAGE_SIZE      : {IMAGE_SIZE}")
print(f"   FEATURE_VERSION : {FEATURE_VERSION}")
print(f"   RANDOM_SEED     : {RANDOM_SEED}")
print(f"   GENERATORS      : {GENERATORS}")
print(f"   FAMILY_ORDER    : {FAMILY_ORDER}")


## Cell 4 — Preview Split CSVs


In [ ]:
SPLITS_DIR = f"{DRIVE_ROOT}/processed/splits"

df = load_splits(SPLITS_DIR)

print(f"Master DataFrame shape : {df.shape}")
print(f"\nColumn names: {list(df.columns)}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nGenerator counts:")
print(df['generator'].value_counts())
print(f"\nSplit counts:")
print(df['split'].value_counts())
print(f"\nlabel_binary counts:")
print(df['label_binary'].value_counts())


In [ ]:
# ── Sanity checks on the DataFrame ──────────────────────────────────────────
assert df['label_binary'].isin([0, 1]).all(), "label_binary must be 0 or 1"
assert df['generator'].isin(GENERATORS).all(), f"Unknown generators found"
assert df['split'].isin(['train', 'val', 'test']).all(), "Unexpected split values"
assert df['filename'].is_unique, "filename column must be unique (no duplicates)"
print("✅ DataFrame sanity checks passed.")


## Cell 5 — Run the Feature Extraction Script

This cell calls the production runner. All extraction logic, logging,
validation, and file saving happens inside `extract_features.py`.

**Expected runtime: 15–30 minutes for 4,800 images.**


In [ ]:
# These paths override the config.py defaults via CLI arguments
SPLITS_DIR_ARG  = f"{DRIVE_ROOT}/processed/splits"
IMAGE_ROOT_ARG  = f"{DRIVE_ROOT}/images"
OUT_ROOT_ARG    = f"{DRIVE_ROOT}/processed/features"
VERSION_ARG     = "v1"   # bump this to create a new versioned run

!python {REPO_DIR}/src/extract_features.py \
    --splits_dir  "{SPLITS_DIR_ARG}" \
    --image_root  "{IMAGE_ROOT_ARG}" \
    --out_root    "{OUT_ROOT_ARG}" \
    --version     {VERSION_ARG}


## Cell 6 — Load and Verify Saved Outputs


In [ ]:
import json
from pathlib import Path

FEAT_DIR  = Path(f"{DRIVE_ROOT}/processed/features/{VERSION_ARG}")
FULL_DIR  = FEAT_DIR / "full"
META_DIR  = FEAT_DIR / "metadata"

# Load arrays
X          = np.load(FULL_DIR / "combined.npy")
labels     = np.load(FULL_DIR / "labels.npy")
generators = np.load(FULL_DIR / "generators.npy")
splits_arr = np.load(FULL_DIR / "splits.npy")

# Load metadata
with open(META_DIR / "feature_metadata.json") as f:
    feat_meta = json.load(f)
with open(META_DIR / "manifest.json") as f:
    manifest = json.load(f)

print("=" * 50)
print("  Post-Extraction Verification")
print("=" * 50)
print(f"combined.npy shape   : {X.shape}")
print(f"labels.npy shape     : {labels.shape}")
print(f"generators.npy shape : {generators.shape}")
print(f"Total dims (manifest): {manifest['total_dims']}")
print(f"NaN count            : {np.isnan(X).sum()}")
print(f"Inf count            : {np.isinf(X).sum()}")
print(f"dtype                : {X.dtype}")
print()
print("Feature family breakdown:")
for name, info in feat_meta['families'].items():
    print(f"  {name:<12} {info['dims']:>6} dims  [{info['start']}:{info['end']}]")
print(f"  {'TOTAL':<12} {feat_meta['total_dims']:>6} dims")
print()
print("Split counts:")
for split in ('train', 'val', 'test'):
    print(f"  {split:<6} : {(splits_arr == split).sum()}")

# Hard assertions
assert X.shape[1] == feat_meta['total_dims'], "Dimension mismatch!"
assert np.isnan(X).sum() == 0, "NaN found in features!"
assert np.isinf(X).sum() == 0, "Inf found in features!"
assert X.dtype == np.float32, "Features are not float32!"
print()
print("✅ All post-extraction assertions passed.")


## Cell 7 — Feature Family Variance Plot


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

feature_variances = X.var(axis=0)

# Compute mean variance per family
family_var_means = {}
for name, info in feat_meta['families'].items():
    family_slice = X[:, info['start']:info['end']+1]
    family_var_means[name] = float(family_slice.var(axis=0).mean())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: per-dimension variance (full combined vector)
axes[0].plot(feature_variances, alpha=0.6, linewidth=0.8, color="#5B9BD5")
axes[0].set_title("Per-Dimension Variance (Combined Vector)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Feature Index")
axes[0].set_ylabel("Variance")
axes[0].grid(True, alpha=0.3)

# Mark family boundaries
colors = plt.cm.Set2.colors
for i, (name, info) in enumerate(feat_meta['families'].items()):
    axes[0].axvline(x=info['start'], color=colors[i % len(colors)],
                    linestyle='--', alpha=0.7, label=name)
axes[0].legend(fontsize=8, loc="upper right")

# Right: mean variance per family
bars = axes[1].bar(
    list(family_var_means.keys()),
    list(family_var_means.values()),
    color=[colors[i % len(colors)] for i in range(len(family_var_means))],
    edgecolor="black", linewidth=0.5, alpha=0.85,
)
axes[1].set_title("Mean Variance per Feature Family", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Mean Variance")
axes[1].set_xlabel("Feature Family")
for bar, val in zip(bars, family_var_means.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                 f"{val:.4f}", ha='center', va='bottom', fontsize=8)
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
fig.savefig(FEAT_DIR / "figures" / "feature_variance.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Variance plot saved.")


## Cell 8 — Class Balance Bar Chart


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: images per generator
gen_counts = {g: (generators == g).sum() for g in GENERATORS}
axes[0].bar(gen_counts.keys(), gen_counts.values(),
            color=colors[:len(GENERATORS)], edgecolor="black", linewidth=0.5, alpha=0.85)
axes[0].set_title("Images per Generator", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].set_xlabel("Generator")
for i, (g, c) in enumerate(gen_counts.items()):
    axes[0].text(i, c + 5, str(c), ha='center', va='bottom', fontsize=9)
axes[0].grid(True, alpha=0.3, axis="y")

# Right: Real vs Fake
label_counts = {"Real (0)": int((labels == 0).sum()), "Fake (1)": int((labels == 1).sum())}
axes[1].bar(label_counts.keys(), label_counts.values(),
            color=["#5B9BD5", "#ED7D31"], edgecolor="black", linewidth=0.5, alpha=0.85)
axes[1].set_title("Binary Class Balance", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Count")
for i, (_, c) in enumerate(label_counts.items()):
    axes[1].text(i, c + 5, str(c), ha='center', va='bottom', fontsize=9)
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
fig.savefig(FEAT_DIR / "figures" / "class_balance.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Class balance plot saved.")


## Cell 9 — PCA 2-D Scatter (Generator Separability)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Scale before PCA (PCA is sensitive to variance differences)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_scaled)

print(f"PCA explained variance ratio: PC1={pca.explained_variance_ratio_[0]:.3f}, "
      f"PC2={pca.explained_variance_ratio_[1]:.3f}")

fig, ax = plt.subplots(figsize=(10, 8))
palette = plt.cm.tab10.colors

for i, gen in enumerate(GENERATORS):
    mask = generators == gen
    ax.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        label=gen, alpha=0.5, s=20,
        color=palette[i], linewidths=0,
    )

ax.set_title(
    f"PCA 2-D Projection by Generator\n"
    f"(PC1={pca.explained_variance_ratio_[0]:.1%}, "
    f"PC2={pca.explained_variance_ratio_[1]:.1%} variance explained)",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("Principal Component 1")
ax.set_ylabel("Principal Component 2")
ax.legend(fontsize=10, markerscale=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FEAT_DIR / "figures" / "pca_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ PCA scatter saved.")


## Cell 10 — Print Feature Report


In [ ]:
report_path = META_DIR / "feature_report.txt"
print(report_path.read_text())


## Summary

After this notebook completes, the following are available on Drive
without ever needing to touch raw images again:

| Experiment | Files needed |
|---|---|
| HOG-only classifier | `train/hog.npy` + `train/labels.npy` |
| Ablation: remove one family | Slice `combined.npy` via `feature_metadata.json` |
| PCA / t-SNE / UMAP | `full/combined.npy` |
| SHAP / Mutual Information | `combined.npy` + `feature_names.txt` |
| Cross-generator transfer matrix | `test/` arrays filtered by `generators.npy` |
